<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="https://www.uoc.edu/content/dam/news/images/noticies/2016/202-nova-marca-uoc.jpg" align="left" width="45%">
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">M2.878 · Trabajo de Fin de Máster · Modelo <i>naive</i></p>
<p style="margin: 0; text-align:right;">2025-2 · Máster universitario en Ciencia de datos</p>
<p style="margin: 0; text-align:right; padding-button: 100px;">Marcos Rodríguez Soler</p>
</div>
</div>
<div style="width:100%;">&nbsp;</div>

# Preprocesamiento de los datos para el modelo _naive_
En este _notebook_ se aplica un modelo _naive_ al conjunto de datos. Esta implementación consiste en la predicción de la demanda futura a lo largo de todo el horizonte temporal como las últimas ventas observadas de cada producto. Este enfoque permite obtener una aproximación muy simple de las ventas futuras, sirviendo como referencia para evaluar el rendimiento de los modelos de aprendizaje automático del proyecto. Adicionalmente, este modelo trabaja con series temporales univariantes, por lo que cada producto del conjunto de datos debe tratarse de forma independiente.

In [ ]:
import random

import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt

from typing import Any, Dict, List

pd.set_option("display.max_columns", None)
%matplotlib inline

<br><br>Dada la simplicidad de este modelo, la implementación se lleva a cabo de forma manual sin librerías externas más allá de _pandas_ o _numpy_. Las predicciones mediante este método se realizan tantos pasos a futuro como se corresponda la duración más larga del ciclo de aprovisionamiento de todos los productos. Este enfoque es el mismo que se ha empleado en los modelos de aprendizaje automático. Asimismo, para que las predicciones se realicen en los mismos elementos de cada serie temporal que se han empleado en los modelos de _machine learning_, la demanda histórica de cada producto se segmenta teniendo en cuenta los 28 días pasados que se utilizan en estos otros modelos mencionados, y también teniendo en cuenta los horizontes de demanda de un máximo de 56 días a futuro. Así pues, el intervalo resultante se parte de forma que el último 20% se correspondería al conjunto de prueba de los modelos de aprendizaje automático.

Por otro lado, en este _notebook_ solo se calculará el RMSE como medida de evaluación del error del modelo _naive_ a través de la función _**rmse()**_ que se define a continuación. El RMSSE en este caso se ignora, ya que esta métrica proporciona el _ratio_ de la suma de errores cuadrados en el conjunto de prueba de un modelo respecto a la suma de errores cuadrados en el conjunto de entrenamiento del mismo modelo _naive_ que se desarrolla en este _notebook_. Se considera que esta métrica no es demasiado informativa en este caso, ya que se estaría comparando el modelo consigo mismo.

In [ ]:
# Funciones para calcular el RMSE y el RMSSE
def rmse(y_real: pd.Series, y_pred: pd.Series) -> float:
    """Devuelve el RMSE de las predicciones de un modelo sobre un conjunto de prueba
    Argumentos:
        y_real: pd.Series -> Demanda real del conjunto de prueba
        y_pred: pd.Series -> Demanda predicha para el conjunto de prueba

    Devuelve
        float -> RMSE
    """
    return np.sqrt(np.mean(np.abs(y_real - y_pred) ** 2))

<br><br>Tras el planteamiento del modelo _naive_, a continuación se incluye el flujo de trabajo para el cálculo de las predicciones de la demanda de cada producto mediante este enfoque.

In [ ]:
# Se importa el conjunto de datos resultante del análisis exploratorio de datos
ruta_dataset: str = "../AED/dataset_preprocesado.csv"
dataset_global: pd.DataFrame = pd.read_csv(ruta_dataset)

In [ ]:
# Se realizan las predicciones del modelo naive y se genera el DataFrame con los resultados
LAG: int = 28
HORIZONTE: int = max(dataset_global["diasLeadtime"] + dataset_global["diasEntrePedidos"])

lista_meta_test: List[pd.DataFrame] = []
lista_y_pred: List[pd.DataFrame] = []
lista_y_real: List[pd.DataFrame] = []

for producto, df_producto in dataset_global.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values(by="fecha").reset_index(drop=True)
    
    y: np.ndarray = df_producto["udsVenta"].values
    
    N: int = len(df_producto)
    N_SEC: int = N - LAG - HORIZONTE + 1
    PARTICION: int = int(N_SEC * 0.8)
    
    meta_test_producto: List[pd.Series] = []
    y_preds_completo: List[List[float]] = []
    y_real_completo: List[np.ndarray] = []
    
    for i in range(PARTICION, N_SEC):
        t: int = i + LAG

        meta_test_producto.append(
            df_producto.iloc[t][[
                "producto", "fecha", "udsVenta", "diasLeadtime", "diasEntrePedidos", "eurPrecioMedio"
            ]]
        )

        ultimo_valor: float = y[t - 1]
        y_pred: List[float] = [ultimo_valor] * HORIZONTE
        y_preds_completo.append(y_pred)

        y_real: np.ndarray = y[t : t + HORIZONTE]
        y_real_completo.append(y_real)
    
    lista_meta_test.append(pd.DataFrame(meta_test_producto))
    
    lista_y_pred.append(
        pd.DataFrame(
            y_preds_completo,
            columns=[f"pred_{h}" for h in range(1, HORIZONTE + 1)]
        )
    )
    
    lista_y_real.append(
        pd.DataFrame(
            y_real_completo,
            columns=[f"real_{h}" for h in range(1, HORIZONTE + 1)]
        )
    )

meta_test: pd.DataFrame = pd.concat(lista_meta_test, ignore_index=True)
y_pred_df: pd.DataFrame = pd.concat(lista_y_pred, ignore_index=True)
y_real_df: pd.DataFrame = pd.concat(lista_y_real, ignore_index=True)

resultados: pd.DataFrame = pd.concat(
    [meta_test.reset_index(drop=True), y_real_df, y_pred_df],
    axis=1
)

resultados["fecha"] = pd.to_datetime(resultados["fecha"])

In [ ]:
# Se calculan los promedios de los errores cometidos en el conjunto de prueba del dataset global
rmses: Dict[str, float] = {
    i: rmse(
        resultados[f"real_{i}"],
        resultados[f"pred_{i}"]
    ) for i in range(1, HORIZONTE + 1)
}

In [ ]:
# Se visualiza la evolución del promedio del RMSE de todos los productos en los horizontes temporales
plt.plot(range(1, HORIZONTE + 1, 1), rmses.values(), color="blue")

plt.title("Evolución del promedio del RMSE de todos los artículos en los horizontes temporales")
plt.xlabel("Días futuros")
plt.xticks(range(1, HORIZONTE + 1, 5))
plt.ylabel("RMSE promedio por producto")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Se calculan los RMSE en el conjunto de prueba del dataset global por producto y se exportan para la evaluación económica
rmses_productos: List[pd.DataFrame] = []

for producto, df_producto in resultados.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")

    rmses_producto: Dict[str, float] = {
        f"hor_{i}": [rmse(
            df_producto[f"real_{i}"],
            df_producto[f"pred_{i}"]
        )] for i in range(1, HORIZONTE + 1)
    }

    rmses_df: pd.DataFrame = pd.DataFrame(rmses_producto)
    rmses_df["producto"] = producto
    rmses_productos.append(rmses_df)

rmses_productos: pd.DataFrame = pd.concat(rmses_productos, ignore_index=True)
rmses_productos.to_csv("rmses_naive_global.csv")

In [ ]:
# Se visualiza la demanda real frente a las predicciones de tres productos aleatorios del dataset global
N_PRODUCTOS: int = 3
N_PREDICCIONES: int = 3
seleccion_productos: List[int] = random.sample(resultados["producto"].unique().tolist(), N_PRODUCTOS)

fig, axs = plt.subplots(nrows=len(seleccion_productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, seleccion_productos):
    res_producto: pd.DataFrame = resultados[resultados["producto"] == producto]
    seleccion_predicciones: List[int] = random.sample(range(len(res_producto)), N_PREDICCIONES)

    fecha_max: str = res_producto["fecha"].max()
    indices_validos: List[int] = [
        i for i in range(len(res_producto))
        if res_producto.iloc[i]["fecha"] + pd.Timedelta(days=HORIZONTE) <= fecha_max
    ]
    seleccion_predicciones: List[int] = random.sample(indices_validos, N_PREDICCIONES)
    
    fechas_real: pd.Series = res_producto["fecha"] + pd.Timedelta(days=1)
    ax.plot(fechas_real, res_producto["real_1"], label="demanda_real", color="blue", alpha=0.5)

    for i, indice_prediccion in enumerate(seleccion_predicciones):
        fila: pd.Series = res_producto.iloc[indice_prediccion]
        predicciones = [fila[f"pred_{h}"] for h in range(1, HORIZONTE + 1)]

        fecha_inicio: str = fila["fecha"]
        fechas_prediccion = [fecha_inicio + pd.Timedelta(days=h) for h in range(1, HORIZONTE + 1)]
        
        ax.plot(fechas_prediccion, predicciones, label=f"prediccion_{i + 1}", linestyle="--", alpha=0.5)

    ax.set_title(f"Demanda real Vs Predicciones de la demanda del producto {producto} del dataset global")
    ax.set_xlabel("Días")
    ax.set_xticks([res_producto["fecha"].iloc[i] for i in range(0, len(res_producto["fecha"]), 20)])
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Unidades")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Se exportan los resultados
resultados.to_csv("res_naive_global.csv")

El modelo _naive_ evidencia un patrón de error periódico con frecuencia semanal, lo que sugiere la presencia de estacionalidad en la demanda. Esta misma característica es captada adecuadamente por los modelos de aprendizaje automático dado que se ha observado una alta importancia de las variables referentes a los días de la semana en los modelos basados en árboles. Consecuentemente, los modelos de _machine learning_ muy posiblemente permitan a la empresa ahorrar una cantidad importante de dinero en comparación con este modelo _naive_, aunque esto se desarrolla en el apartado **Evaluación del Impacto Económico**.